# Klasifikace obrázků CIFAR-10 pomocí konvoluční sítě

- Úkolem cvičení je natrénovat konvoluční síť pro klasifikaci na datasetu CIFAR-10 s alespoň 75% úpěšností na validační sadě.
- Nejprve implementujeme konvoluci jako diferencovatelnou operaci a vrstvu.
- Poté přidáme max-pooling a reshape.
- Z těchto funkčních bloků nakonec sestavíme konvoluční síť.
- Dobře navržená síť dosáhne i přes 90 % validační accuracy.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')  # import tests

import torch
import torchvision

import ans
from tests import test_convolution

In [ ]:
torch.set_printoptions(profile='short')

# Dvourozměrná konvoluce jako diferencovatelná operace

- Konvoluci implementujeme jako `ans.nn.Function` i jako `ans.nn.Module`.

**Parametry**

| atribut  | typ                               | značení          | rozměr                         | poznámka                                             |
|----------|-----------------------------------|------------------|--------------------------------|------------------------------------------------------|
| `weight` | `ans.autograd.Variable`           | $\boldsymbol{w}$ | $F \times C \times K \times K$ | `num_filters x in_chnls x kernel_size x kernel_size` |
| `bias`   | `Optional[ans.autograd.Variable]` | $\boldsymbol{b}$ | $F$                            | nepovinný                                            |

**Inicializace**

- *Váhy* inicializujte opět uniformní variantou Xavier/Glorot/Kaiming/He, tak, [jak je tomu v knihovně v PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
  $$
  w_{f,c,i,j} \sim \mathcal{U}\left(\frac{-1}{\sqrt{C\cdot K^2}}, \frac{+1}{\sqrt{C\cdot K^2}}\right)
  $$
- *Bias* bude *nepovinný* parametr a příp. jej inicializujeme na nuly. Výsledný vektor musí být atribut typu `Variable`, příp. objekt `None`, pokud je do `__init__` předáno `bias=False`.

**Dopředný průchod**

$$
\boldsymbol{z}_n = \textrm{conv2}\left( \boldsymbol{x}_n, \boldsymbol{w}, \boldsymbol{b}; S, P, D \right)
$$
kde
- $\boldsymbol{x}_n$ je $n$-tý "obrázek" dávky jako tensor s rozměry $C \times H \times W$
- $S$ (hyperparametr) je krok (stride)
- $P$ (hyperparametr) je padding
- $D$ (hyperparametr) je dilation
- $\boldsymbol{z}_n$ je $n$-tý výstup dávky jako tensor s rozměry $F \times H' \times W'$
- Pro rozměr výstupu $W'$ platí
  $$
  W' = \left\lfloor \frac{W + 2 \cdot P - K}{S} \right\rfloor + 1
  $$

**Zpětný průchod**
$$
\begin{split}
    \overline{\boldsymbol{x}}_n & = \textrm{conv2}^\top(\overline{\boldsymbol{z}}_n, \boldsymbol{w}, \boldsymbol{0}; S, P, D) \\
    \boldsymbol{\overline{w}}_{f,c} & = \sum_{n=1}^N{ \textrm{conv2}(\boldsymbol{x}_{n,c}, \overline{\boldsymbol{z}}_{n,f}, \boldsymbol{0}; D, P, S) }
\end{split}
$$
- kde 
  - $\textrm{conv2}^\top(\cdot)$ je tzv. transponovaná dvourozměrná konvoluce
  - $\boldsymbol{x}_{n,c}$ je $c$-tý kanál $n$-tého vstupu jako matice s rozměry $H \times W$
  - $\overline{\boldsymbol{z}}_{n,f}$ je příchozí gradient na $f$-tý kanál $n$-tého výstupu jako tensor s rozměry $H' \times W'$
  - $\boldsymbol{\overline{w}}_{f,c}$ je vypočítáný gradient na $c$-tý kanál $f$-tého filtru jako matice s rozměry $K \times K$

**Implementace**

- Vzhledem k výpočetní náročnosti nebudeme funkce $\textrm{conv2}(\cdot)$ a $\textrm{conv2}^\top(\cdot)$ implementovat vlastními silami pomocí for cyklů ani komplikovaného broadcastingu, ale použijeme funkce `conv2d`, resp. `conv_transpose2d` modulu `torch.nn.functional` knihovny Pytorch.
- Není to vyžadováno, ale gradient na váhy $\boldsymbol{\overline{w}}$ lze získat i zcela bez použití cyklů (pro `groups=1`).

**Output padding u `conv_transpose2d` pro výpočet gradientu na vstup $\overline{\boldsymbol{x}}_n$**

- Pokud je stride $S \gt 1$, může se stát, že z různě velkých vstupů $\boldsymbol{x}_n$, $\boldsymbol{x}_m$ funkcí vzniknou stejně velké výstupy $\boldsymbol{z}_n$, resp. $\boldsymbol{z}_m$.
- Pokud potom použijeme $\textrm{conv2}^\top$ jako zpětný průchod pro výpočet gradientu na vstup $\overline{\boldsymbol{x}}_n$, není rozměr výsledného gradientu na vstup jednoznačně určená, a tak $\overline{\boldsymbol{x}}_n$ nemusí svou velikostí přesně odpovídat $\boldsymbol{x}$.
- Funkce `torch.nn.conv_transpose2d` proto zavádí argument `output_padding`, který výstup nastaví o požadovanou hodnotu.
- Aby měl $\overline{\boldsymbol{x}}_n$ rozměry shodné s $\boldsymbol{x}_n$ a procházely všechny testy, je nutné `output_padding` dopočítat jako
  $$
  \begin{split}
    O_x &= W - (W' - 1) \cdot S + 2 \cdot P - (K - 1) \cdot D - 1 \\
    O_y &= H - (H' - 1) \cdot S + 2 \cdot P - (K - 1) \cdot D - 1
  \end{split}
  $$

**Oříznutí gradientu na váhy $\boldsymbol{\overline{w}}$**

- Může se stát, že výstup konvoluce $\boldsymbol{\overline{w}}_{f,c} = \textrm{conv2}(\boldsymbol{x}_{n,c}, \overline{\boldsymbol{z}}_{n,f}, \ldots)$ vyjde větší než $K \times K$.
- V takovém případě stačí výsledek o přebytečné hodnoty "zdola" a "zprava" oříznout.

### TODO: implementujte třídu `ans.nn.Conv2d`

In [ ]:
test_convolution.TestConv2dFunction.eval()

In [ ]:
test_convolution.TestConv2dFunctionOptional.eval()

### TODO: implementujte třídu `ans.nn.Conv2d`

In [ ]:
test_convolution.TestConv2dModule.eval()

# Max-pooling

- Téměř nedílnou součástí konvolučních sítí jsou tzv. [pooling vrstvy](https://d2l.ai/chapter_convolutional-neural-networks/pooling.html).
- Implementujeme si jednu z nejčastějších, tzv. max-pooling.

**Dopředný průchod**

- Vstup $\boldsymbol{x}_n$ rozdělíme na $K \times K$ velké nepřekrávající se oblasti a z každé z nich vypočteme maximum.
- Abychom replikovali chování `MaxPool2d` knihovny, pro velikost výstupu bude platit
  $$
  W' = \left\lfloor\frac{W}{K}\right\rfloor
  $$
  tj. případech, kdy velikost vstupu $W$ není dělitelná velikostí okna $K$, nadbytečné hodnoty ve vstupu zahodíme.

**Zpětný průchod**

- Gradient na vstup je
  - 1 na pozicích odpovídajích pozicím maxim,
  - 0 na ostatních pozicích.

**Implementace a tipy**

- Oproti možnostem v knihovně PyTorch si redukujeme počet hyperparametrů a ponecháme pouze velikost pooling okolí $K$ pod jménem `kernel_size`.
- Tento parametr bude celé číslo, tzn. že se omezíme na čtvercové okno o roměru $K \times K$.
- *Operaci naprogramujte vektorově bez použití cyklů!*
  - Např. max-pooling vektoru s velikostí okna `kernel_size=2`:
    ``` python
    >>> x = torch.tensor([1, 2, 4, 3, 5, 5])
    >>> x.reshape(3, 2).max(dim=1)
    (tensor([2, 4, 5]), tensor([1, 0, 0]))
    ```
  - U dvourozměrného vstupu je potřeba obdobný reshape provést pro výšku i šířku.
  - Hodit se může také funkce `torch.transpose`, pomocí které lze posunout přidané dimenze na poslední dvě místa, což umožní reshape na `(..., kernel_size * kernel_size)`.

### TODO: implementujte třídu `ans.nn.MaxPool2dFunction`

In [ ]:
test_convolution.TestMaxPool2dFunction.eval()

### TODO: implementujte třídu `ans.nn.MaxPool2d`

In [ ]:
test_convolution.TestMaxPool2dModule.eval()

# Operace `Variable.reshape` a vrstva `Flatten`

- Výstup konvoluce je tvaru $N \times C \times H \times W$.
- Abychom mohli klasifikovat lineární vrstvou, potřebujeme reshape na $N \times D$, tj. potřebujeme výstup "zploštit" (flatten).
- Operaci implementujeme obecněji jako libovolný reshape a to přímo do třídy `Variable`.
- Zpětný průchod je reshape z $N \times D$ zpět na $N \times C \times H \times W$.
- Vrstva `Flatten` pak provede speciální případ reshape $N \times D_1 \times D_2 \times \ldots \rightarrow N \times D$, kde $D = D_1 \cdot D_2 \cdot \ldots$.

### TODO: implementujte metodu `ans.autograd.Variable.reshape`

In [ ]:
test_convolution.TestReshapeVariable.eval()

### TODO: implementujte třídu `ans.nn.Flatten`

In [ ]:
test_convolution.TestFlattenModule.eval()

# (Bonus) Batch normalizace pro 2D vstupy

- Pokud máte v knihovně `ans.nn` implementovanou normalizaci dávky pro 1D vstupy z minulého cvičení, můžete ji poměrně jednoduše rozšířit pro 2D vstupy.
- Tzv. [spatial batchnorm](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html), jak se často 2D varianta normalizace dávky označuje v anglické literatuře, *normalizuje přes jednotlivé kanály vstupu*.
- To znamená, že vektory $\boldsymbol{\mu}, \boldsymbol{\sigma^2}, \boldsymbol{\gamma}, \boldsymbol{\beta}, \boldsymbol{m}, \boldsymbol{v}^2$ jsou vektory o rozměru $C$.
- Operaci lze naimplementovat následovně:
  1. převedení vstupního tensoru $\boldsymbol{x}$ s rozměry $N \times C \times H \times W$ na 2D matici $(N \cdot H \cdot W) \times C$,
  2. aplikace 1D batch normalizace
  3. převedení výstupní matice $(N \cdot H \cdot W) \times C$ zpět do tensoru s rozměry $N \times C \times H \times W$.

### TODO: implementujte třídu `ans.nn.BatchNorm2dFunction`

In [ ]:
test_convolution.TestBatchNorm2dFunction.eval()

### TODO: implementujte třídu `ans.nn.BatchNorm2d`

In [ ]:
test_convolution.TestBatchNorm2dModule.eval()

# Konvoluční síť

- Konvoluční síť implementuje jako třídu `Backbone(ans.nn.Module)` a bude sloužit jako "páteř" (backbone) pro `ans.classification.AutogradClassifier`.
- Třída bude mít na vstupu dávku obrázků $N \times C \times H \times W$ a vrátí dávku logitů $N \times K$.
- Aby procházely testy, backbone musí mít
  - `__init__(self, in_channels: int, num_classes: int, **kwargs)`, tj. dva poziční parametry
    1. `in_channels` ... počet kanálů $C$ na vstupu (např. 3 pro RGB)
    2. `num_classes` ... počet tříd $K$ na výstupu
  - alespoň jednu konvoluční vrstvu `ans.nn.Conv2d`,
  - `forward` využívající konvoluční vrstvu.
- Jelikož hledání ideální konvoluční architektury metodou pokus omyl by mohlo být časově náročné, můžete zkusit např. jednu z následujících
  
  | název   | architektura                                                                                                             |
  |---------|--------------------------------------------------------------------------------------------------------------------------|
  | VGG7    | `CR(64), M(2), CR(128), M(2), CR(256), M(2), CR(512), M(2), CR(512), M(2), LR(512), L(10)`                               |
  | VGG7BN  | `CNR(64), M(2), CNR(128), M(2), CNR(256), M(2), CNR(512), M(2), CNR(512), M(2), LR(512), L(10)`                          |
  | ResNet9 | `CNR(64), CNR(128), M(2), res(CNR(128), CNR(128)), CNR(256), M(2), CNR(512), M(2), res(CNR(512), CNR(512)), M(4), L(10)` |
  
  kde:
  - `CR(64)` znamená dvourozměrnou konvoluci (`C`) se 64 filtry a biasy následovanou ReLU (`R`); všechny konvoluce mají velikost 3x3,
  - `CNR(64)` znamená blok konvoluce (`C`) - batch normalizace (`N`) - ReLU (`R`),
  - `M(2)` znamená max-pooling s oknem o velikosti 2 a krokem 2
  - `LR(512)` znamená lineární vrstvu (`L`) následovanou ReLU (`R`) s *výstupním* vektorem o rozměru 512
  - `res(convblock)` značí residuální blok formy `z = x + convblock(x)`
- Architektury jsou pouze doporučené. Můžete vyvinout i libovolnou jinou.
- VGG7 je klasická konv. síť inspirovaná architekturou [VGG](https://arxiv.org/abs/1409.1556), je pouze zmenšená na 7 vrstev. VGG7BN k ní přidává batch normalizaci.
- ResNet9 je architektura, která se dobře umisťuje v benchmarku [Standford DAWNBench](https://dawnd9.sites.stanford.edu/dawnbench) v žebříčku hodnotícím trénovací čas pro dosažení 94 % validační přesnosti. Byla vyvinuta v sérii příspěvků [How to train your ResNet](https://myrtle.ai/learn/how-to-train-your-resnet/), která se zabývala trénováním na CIFAR10. Série již bohužel není dostupná.

### TODO: Vytvořte model konvoluční sítě

In [ ]:
class Backbone(ans.nn.Module):

    def __init__(self, in_channels: int, num_classes: int, **kwargs) -> None:
        super().__init__()

        ########################################
        # TODO: implement

        raise NotImplementedError
    
        # ENDTODO
        ########################################
    
    def forward(self, x: ans.autograd.Variable) -> ans.autograd.Variable:
        ########################################
        # TODO: implement

        raise NotImplementedError
        
        # ENDTODO
        ########################################

In [ ]:
test_convolution.TestBackbone.eval(backbone_cls=Backbone)

## Preprocessing

- Požadavky na a způsob preprocessingu/augmentace jsou téměř totožné s cvičením [`neural_library`](neural_library.ipynb).
- Jediný rozdíl je, že data vstupující do sítě musejí mít tvar $N \times C \times H \times W$.

In [ ]:
class DataPreprocessor:

    def fit(self, dataset: torchvision.datasets.CIFAR10) -> None:
        ########################################
        # TODO: implement if needed
    
        pass
    
        # ENDTODO
        ########################################

    def transform(self, dataset: torchvision.datasets.CIFAR10, train: bool = False) -> torch.utils.data.TensorDataset:
        ########################################
        # TODO: implement

        raise NotImplementedError

        # ENDTODO
        ########################################

        return torch.utils.data.TensorDataset(x, y)

In [ ]:
test_convolution.TestDataPreprocessor.eval(preprocessor_cls=DataPreprocessor)

# Trénování klasifikátoru

- Pro klasifikaci použijte `ans.classification.AutogradClassifier` z minula, kterému jako `backbone` předáte konvoluční "páteř" definovanou výše, např.
  ``` python
  backbone = Backbone(3, 10)
  optimizer = ans.nn.SGD(backbone.parameters(), learning_rate=learning_rate)
  model = ans.classification.AutogradClassifier(backbone, optimizer)
  ```
- Nejlepší model uložte jako
  ``` python
  model.save('../output/conv_classifier.pt')
  ```
- Pro trénování budete velmi pravděpodobně potřebovat GPU. Na CPU běží konvoluční sítě bohužel velmi pomalu.

### TODO: Natrénujte *konvoluční* klasifikátor, který dosáhne alespoň 75 % (bonusově **85 %** a **90 %**) *validační* accuracy.

In [ ]:
ans.utils.seed_everything(0)

# ...
device = 'cuda'

# dataset
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True)
val_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True)

# preprocessing
preprocessor = DataPreprocessor()
preprocessor.fit(train_dataset)
train_dataset = preprocessor.transform(train_dataset, train=True)
val_dataset = preprocessor.transform(val_dataset, train=False)

# loadery
train_loader = ans.data.BatchLoader(train_dataset, batch_size=..., shuffle=True, device=device)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=..., shuffle=False, device=device)

# model
# backbone = ...
# backbone = backbone.to(device=device)
# optimizer = ...
# model = ans.classification.AutogradClassifier(backbone, optimizer)

# ...

In [ ]:
test_convolution.TestValAccuracy75.eval(preprocessor_cls=DataPreprocessor)

In [ ]:
test_convolution.TestValAccuracy85.eval(preprocessor_cls=DataPreprocessor)

In [ ]:
test_convolution.TestValAccuracy90.eval(preprocessor_cls=DataPreprocessor)